In [2]:
import pandas as pd
from tqdm.auto import tqdm
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns
from matplotlib.colors import to_rgba

tqdm.pandas()

c:\Users\erikm\OneDrive\Documents\Github\STATS 5000 PROJ\CU_STAT5000_Final_Proj\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Load the Data

In [3]:
spotify_tracks = pd.read_csv('spotify_tracks_pop.csv')
display(pd.read_csv('spotify_tracks_pop.csv').head(5))
spotify_tracks['track_album_release_date'] = pd.to_datetime(spotify_tracks['track_album_release_date'])
spotify_tracks.shape


,id,energy,tempo,danceability,loudness,liveness,valence,time_signature,speechiness,track_album_release_date,...,key,duration_min,acousticness,first_listed_artist,track_album_name,playlist_name,playlist_genre,playlist_subgenre,track_name,track_popularity
0,2plbrEY59IikOBgBGLjaoe,0.592,157.969,0.521,-7.777,0.122,0.535,3/4,0.0304,2024-08-16,...,6.0,4.194467,0.3080,Lady Gaga,Die With A Smile,Today's Top Hits,pop,mainstream,Die With A Smile,100
1,2plbrEY59IikOBgBGLjaoe,0.592,157.969,0.521,-7.777,0.122,0.535,3/4,0.0304,2024-08-16,...,6.0,4.194467,0.3080,Lady Gaga,Die With A Smile,Global Top 50,pop,global,Die With A Smile,100
2,2plbrEY59IikOBgBGLjaoe,0.592,157.969,0.521,-7.777,0.122,0.535,3/4,0.0304,2024-08-16,...,6.0,4.194467,0.3080,Lady Gaga,Die With A Smile,Top Gaming Tracks,gaming,modern,Die With A Smile,100
3,5vNRhkKd0yEAg8suGBpjeY,0.783,149.027,0.777,-4.477,0.355,0.939,4/4,0.2600,2024-10-18,...,0.0,2.831950,0.0283,ROSÉ,APT.,Today's Top Hits,pop,mainstream,APT.,98
4,5vNRhkKd0yEAg8suGBpjeY,0.783,149.027,0.777,-4.477,0.355,0.939,4/4,0.2600,2024-10-18,...,0.0,2.831950,0.0283,ROSÉ,APT.,Global Top 50,pop,global,APT.,98


(4684, 22)

Question: Is there evidence of a difference in average popularity between Major and Minor tracks

Creating a new data set of of genres with modes greater than or equal to 20 and removing duplicates in our newly constructed dataset to ensure unadulterated statistical evaluation

In [11]:
genre_mode_counts = pd.crosstab(
    spotify_tracks['playlist_genre'], spotify_tracks['mode'])

eligible_genres = genre_mode_counts[
    (genre_mode_counts["Major"] >=20) | (genre_mode_counts["Minor"] >=20)].index

eligible_genre_data = spotify_tracks[
    spotify_tracks['playlist_genre'].isin(eligible_genres)]
eligible_genre_data = eligible_genre_data.drop_duplicates(subset=["id"])

In [12]:
mode_counts = (
    eligible_genre_data
    .groupby(["playlist_genre", "mode"])
    .size()
    .unstack(fill_value=0)
)

display(mode_counts)

mode,Major,Minor
playlist_genre,,
afrobeats,35,31
ambient,199,137
arabic,87,91
blues,40,32
brazilian,78,54
classical,56,55
electronic,269,272
folk,54,10
gaming,62,49


In [13]:
mode_summary = (
    eligible_genre_data
    .groupby(["playlist_genre", "mode"])["track_popularity"]
    .agg(["mean", "std", "count"])
    .reset_index()
)
display(mode_summary)

,playlist_genre,mode,mean,std,count
0,afrobeats,Major,51.228571,16.782494,35
1,afrobeats,Minor,45.290323,18.841786,31
2,ambient,Major,49.804020,16.165395,199
3,ambient,Minor,48.510949,15.700841,137
4,arabic,Major,47.643678,14.567387,87
5,arabic,Minor,50.890110,15.706651,91
6,blues,Major,58.875000,18.659139,40
7,blues,Minor,61.781250,18.438624,32
8,brazilian,Major,37.897436,17.842449,78
9,brazilian,Minor,40.611111,18.102868,54


Performing a two-sided Welch's t-test to compare the mean popularity of Major and Minor songs using a significance level of $\alpha = 0.05$.
\begin{align}
H_o: \mu_{Major} = \mu_{minor}\\
H_1: \mu_{Major} \ne \mu_{minor}\\
\end{align}

In [14]:
from scipy.stats import ttest_ind

def welch_ttest(data, genre):
    major = data[
        (data["playlist_genre"] == genre) &
        (data["mode"] == "Major")
    ]["track_popularity"]

    minor = data[
        (data["playlist_genre"] == genre) &
        (data["mode"] == "Minor")
    ]["track_popularity"]

    t_stat, p_value = ttest_ind(major, minor, equal_var=False)
    return {
        "genre": genre,
        "t_stat": t_stat,
        "p_value": p_value,
        "major_mean": major.mean(),
        "minor_mean": minor.mean(),
        "difference in means": major.mean() - minor.mean(),
        "t_stat": t_stat,
        
    }

In [15]:
genres = eligible_genre_data["playlist_genre"].unique()
welch_results = pd.DataFrame([welch_ttest(eligible_genre_data, genre) for genre in genres])
welch_results["significant"] = welch_results["p_value"] < 0.05 
display(welch_results)

,genre,t_stat,p_value,major_mean,minor_mean,difference in means,significant
0,pop,0.814167,0.416207,62.397849,60.256579,2.141271,False
1,latin,-1.136926,0.256280,53.686486,56.122066,-2.435579,False
2,folk,0.910547,0.379806,60.018519,54.000000,6.018519,False
3,hip-hop,0.243402,0.807845,62.169399,61.696970,0.472429,False
4,gaming,-0.481180,0.631355,69.983871,71.510204,-1.526333,False
5,ambient,0.732932,0.464176,49.804020,48.510949,1.293071,False
6,r&b,1.311963,0.196501,77.214286,75.588235,1.626050,False
7,rock,1.778417,0.077595,66.239583,62.265060,3.974523,False
8,electronic,-3.190602,0.001504,48.773234,53.558824,-4.785589,True
9,punk,-0.096992,0.923183,66.446809,66.750000,-0.303191,False


Now we will perform a $95\%$ bootstrap confidence interval for each genre to see estimate the uncertainty around each difference in means.

In [16]:
def bootstrap_difference(data, genre, n_iterations = 10000):
    major = data[
        (data["playlist_genre"] == genre) &
        (data["mode"] == "Major")
    ]["track_popularity"]

    minor = data[
        (data["playlist_genre"] == genre) &
        (data["mode"] == "Minor")
    ]["track_popularity"]

    differences = []
    for _ in range(n_iterations):
        major_sample = np.random.choice(major, size=len(major), replace=True)
        minor_sample = np.random.choice(minor, size=len(minor), replace=True)
        differences.append(major_sample.mean() - minor_sample.mean())

    lower_bound = np.percentile(differences, 2.5)
    upper_bound = np.percentile(differences, 97.5)

    return {
        "genre": genre,
        "lower_bound": lower_bound,
        "upper_bound": upper_bound
    }

In [17]:
boostrap_results = []
for genre in genres:
    result = bootstrap_difference(eligible_genre_data, genre)
    boostrap_results.append(result)

boostrap_results = pd.DataFrame(boostrap_results)
boostrap_results["significant"] = (boostrap_results["lower_bound"] > 0) | (boostrap_results["upper_bound"] < 0)
display(boostrap_results)

,genre,lower_bound,upper_bound,significant
0,pop,-2.994558,7.256550,False
1,latin,-6.616545,1.807213,False
2,folk,-5.918704,18.752315,False
3,hip-hop,-3.342707,4.223828,False
4,gaming,-7.713693,4.641935,False
5,ambient,-2.100921,4.687712,False
6,r&b,-0.760767,3.989601,False
7,rock,-0.245777,8.455433,False
8,electronic,-7.709989,-1.886384,True
9,punk,-6.139415,5.803218,False
